# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 4: Prompt Engineering</font>

# <font color="#003660">Notebook 1: Basics of Prompt Engineering</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... understand that a language model sees one long sequence of tokens and continues it, <br>
        ... can prompt a base model with zero-, one-, and few-shot prompts and with chain-of-thought examples, <br>
        ... know the three reasons why a generation stops, <br>
        ... can build a prompt template for a task and measure how well it works.
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:

* [Raschka (2024): Building a Large Language Model (From Scratch)](https://www.manning.com/books/build-a-large-language-model-from-scratch)
* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [Prompt Engineering Guide](https://www.promptingguide.ai/)

These documentations can help you for your projects:
* Decoding strategies: [Hugging Face (2025): Generation strategies](https://huggingface.co/docs/transformers/generation_strategies)
* Chat templates: [Hugging Face: Chat templates](https://huggingface.co/docs/transformers/chat_templating)
* Ollama API: [Ollama API documentation](https://docs.ollama.com/api)
* OpenAI API: [OpenAI Python SDK](https://platform.openai.com/docs/api-reference/chat/create?lang=python)
* Ollama provides OpenAI API compatible endpoints: [Ollama](https://github.com/ollama/ollama), [Ollama Python](https://github.com/ollama/ollama-python), [Ollama OpenAI Compatibility](https://docs.ollama.com/api/openai-compatibility)
* HuggingFace provides OpenAI API compatible endpoints: [HuggingFace Inference Providers](https://huggingface.co/docs/inference-providers/index?python-clients=openai&javascript-clients=openai), [HuggingFace Endpoints](https://huggingface.co/docs/inference-endpoints/index)

## From Session 3 to the Continuation Machine

In Session 3 you generated text with a decoder model step by step ([3.01 - Writing with Transformers](https://github.com/olivermueller/amlta-2026/blob/main/Session_03/3.01%20-%20Writing%20with%20Transformers.ipynb)). Keep these ideas separate:

* **Tokenization** turns the text into token IDs, and the chosen token ID back into text.
* **Next Token (Word) Prediction** scores every token and turns the scores into the next-token probability distribution.
* **Decoding** applies a rule, such as greedy decoding, to choose from that distribution.
* **Autoregressive Generation** puts this process in a loop.

If you want to repeat the decoding strategies, the Hugging Face documentation has a good overview: [Generation strategies](https://huggingface.co/docs/transformers/generation_strategies).

From now on, think of a language model as a large **continuation machine**: you give it the beginning of a text, and it plausibly completes it. This is what an LLM does at a high level, using the probability function it learned during training. Everything we do with prompts in this session builds on this one idea.

But first we need a small setup of a magic tool called **Ollama**.

## Setup: Ollama

[Ollama](https://ollama.com/) is a free, open-source tool that downloads language models and runs them on your own computer. It starts a local server to which your code can send prompts. People use it to try out open models, to build prototypes, and to work with models when data must not leave their own machine.

The setup has four steps: install Ollama, start the server, pull the model, and connect to the server from Python.

### Step 1: Install Ollama

The cell below installs Ollama on Linux, for example on Google Colab. It first installs `pciutils`, which the Ollama installer uses to detect your GPU, and `zstd`, which it needs to unpack the download. On macOS or Windows, download the app from [ollama.com/download](https://ollama.com/download) instead and skip this cell.

In [ ]:
!sudo apt-get install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

### Step 2: Start the Ollama server

The installer may say that the server has started, but on Colab it has not. Let's start it ourselves with the code below. The server keeps running in the background while you work with this notebook. On macOS and Windows, the Ollama app starts the server for you.

In [ ]:
import subprocess
import time

# the server writes its log into a file; a pipe that nobody reads could fill up and block the server
ollama_log = open("ollama.log", "w")
process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)

# give the server a few seconds to start
time.sleep(5)

### Step 3: Pull the model

Now we download ("pull") our first model, Llama 3.2 1B. The `text` in its name means that this is a **base model**: it was only trained to continue text. What that means is the main topic of this notebook.

In [ ]:
# the download takes about a minute
!ollama pull llama3.2:1b-text-q4_K_M

BASE_MODEL = "llama3.2:1b-text-q4_K_M"  # base model: only continues text

We have now pulled our first model. But do not confuse it with *large* language models: Llama 3.2 1B has about 1.2 billion parameters, while large language models have tens to hundreds of billions. Models of this size are called **small language models (SLMs)**, and they even run on a laptop. If you want to know more about SLMs, read [Wang et al. (2025)](https://doi.org/10.1145/3711896.3736563). We need it for educational purposes only ;)

### Step 4: Connect from Python

We install Ollama's Python package, `ollama`, and create a client that talks to our local server. To check the connection, the server lists the model we pulled.

Why don't we use Hugging Face, as in Session 3? Projects like Ollama and vLLM are built for **inference**: they serve models fast and memory-efficiently, even to many users at once. Hugging Face `transformers` is the better tool to look inside a model and to train it. We will come back to Hugging Face when we fine-tune LLMs.

In [ ]:
!pip install ollama

import ollama

olm = ollama.Client(host="http://127.0.0.1:11434")  # address of our local Ollama server
for model in olm.list().models:
    print(model.model)

## Part 1: The Continuation Machine

For a base model, there is nothing between your text and the model: no formatting, no roles, no special tokens. The text is split into tokens, and the model continues it.

We send the text with `olm.generate()`. The `options` control the decoding from Session 3: `num_predict` is the maximum number of new tokens, and a `temperature` of 0 means greedy decoding. Let's give it the beginning of a poem ("Twinkle, Twinkle, Little Star" by Jane Taylor, 1806).

In [ ]:
poem = """Twinkle, twinkle, little star,
How I wonder what you are!
Up above the world so high,
Like a diamond in the sky.

When the blazing sun is gone,
When he nothing shines upon,
Then you show your little light,
Twinkle, twinkle, all the night.

"""
response = olm.generate(model=BASE_MODEL, prompt=poem, options={"num_predict": 60, "temperature": 0})
print(response.response)

The model continues the text in the shape of the poem: same line length, same rhythm, often even the same refrain. It does not look anything up; it produces a *plausible* continuation, token by token, with the decoding rules from Session 3.

The response contains more than the text, for example how many tokens the prompt had, how many tokens the model generated, and why it stopped:

In [ ]:
print("tokens in the prompt:", response.prompt_eval_count)
print("tokens generated:", response.eval_count)
print("stopped because of:", response.done_reason)

`length` means that the model reached `num_predict`. More about stopping in Part 4.

And what did Ollama do with our text before the model saw it? Every model in Ollama has a **template** that wraps the prompt. For our base model, the template is only `{{ .Prompt }}`: the prompt itself, nothing added. What you send is exactly what the model sees.

In [ ]:
print(olm.show(BASE_MODEL).template)

### Your turn

Before you run the next cell, **predict**: which of the two prompts gives you an answer about Einstein, and what will the model do with the other one?

In [ ]:
prompts = ["What were the main achievements of Albert Einstein?", "The main achievements of Albert Einstein were"]

for prompt in prompts:
    response = olm.generate(model=BASE_MODEL, prompt=prompt, options={"num_predict": 80, "temperature": 0})
    print(prompt + response.response)
    print("-" * 40)

<details>
<summary><b>Explanation</b> (open after running the cells)</summary>

The first prompt is a question, but the base model does not answer questions. It treats the question like the title of a document and continues with something that could follow such a title: a list, more questions, or an unrelated text. The second prompt is the *beginning of an answer*, so the most likely continuation reads like an answer ([Liu et al., 2023](https://doi.org/10.1145/3560815)). Also check whether everything the model writes is correct: fluent text is not the same as true text.
</details>

## Part 2: Prompting a Base Model: Zero-, One-, and Few-Shot

**Prompting** can be referred to as passing downstream tasks as textual prompts, unambiguous instructions reformulated to solve like the training data, to LLMs without further retraining while **Prompt (template) engineering** denotes the development of the most appropriate prompt to solve a task ([Kaltenpoth and Müller, 2024](https://aisel.aisnet.org/wi2024/91/); [Liu et al., 2023](https://doi.org/10.1145/3560815)).

A base model continues text. To make it solve a task, we write the *beginning of a text* whose most likely continuation is the answer. The simplest way is to just ask. This is called **zero-shot prompting** ([Radford et al., 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)): no examples, only the task.

In [ ]:
question = """When did world war 2 start?"""

qa_prompt = """Q: When did world war 2 start?
A:"""

for prompt in [question, qa_prompt]:
    response = olm.generate(model=BASE_MODEL, prompt=prompt, options={"num_predict": 40, "temperature": 0})
    print(prompt + response.response)
    print("-" * 40)

The question alone rarely gives a clean answer. The `Q:`/`A:` format tells the model that an answer follows, because in its training data an `A:` usually follows a `Q:`.

Note that the prompt ends with `A:` and not with `A: `. Most tokenizers attach a space to the *following* word (` 1939` is one token), so a trailing space would leave the model with an unusual token sequence.

Let's try a classification task: the sentiment of a short movie review. First without an example, then with one example. The second version is called **one-shot prompting** ([Brown et al., 2020](https://papers.nips.cc/paper_files/paper/2020/file/1457c0d6bfcb4967418bfb8ac142f64a-Paper.pdf)).

In [ ]:
zero_shot_prompt = """'I love this movie' is a"""

one_shot_prompt = """'I hate this movie' is a negative comment.
'I love this movie' is a"""

for prompt in [zero_shot_prompt, one_shot_prompt]:
    response = olm.generate(model=BASE_MODEL, prompt=prompt, options={"num_predict": 20, "temperature": 0})
    print(prompt + response.response)
    print("-" * 40)

With several examples, it is called **few-shot prompting** ([Liu et al., 2023](https://doi.org/10.1145/3560815); [Brown et al., 2020](https://papers.nips.cc/paper_files/paper/2020/file/1457c0d6bfcb4967418bfb8ac142f64a-Paper.pdf)). Here we combine it with the `Q:`/`A:` format:

In [ ]:
few_shot_prompt = """Q: What is 11x2/2? A: 11
Q: What is 11x4/2? A: 22
Q: What is 11x6/2? A: 33
Q: What is 11x8/2? A:"""

response = olm.generate(model=BASE_MODEL, prompt=few_shot_prompt, options={"num_predict": 40, "temperature": 0, "seed": 42})
print(response.response)

### Your turn

1. Look at the output of the few-shot cell again. **What does the model write after the answer, and why?**

<details>
<summary><b>Explanation for question 1</b></summary>

The model keeps going and invents new questions and answers. For the model, the most likely continuation of a list of `Q:`/`A:` pairs is another `Q:`/`A:` pair. Nothing in the text tells it to stop. Remember this for Part 4.
</details>

2. **A real task: sentiment classification.** Below are ten short movie reviews with their sentiment. A **prompt template** is a prompt with a placeholder, here `{text}`, that is filled in with each input. `score()` fills in every review, sends the prompt to the model, and checks the model's answer. It also prints the prompt for the first review, so you see exactly what the model sees. Run the next cell once to define `score()`.
    * Run the zero-shot template below and look at the model's answers.
    * Write a one-shot and a few-shot template. Use your own example reviews, not reviews from the list, otherwise the model only has to copy the answer.
    * Which template scores best? Compare your score with your neighbours: same model, same reviews, different prompts.

In [ ]:
import re

def check(answer, expected):
    """Looks for the expected answer behind "The answer is" or, if the model did not write that, in the first line."""
    position = answer.lower().find("answer is")
    if position >= 0:
        answer = answer[position + len("answer is"):]
    first_line = next((line.strip() for line in answer.splitlines() if line.strip()), "")
    correct = re.search(rf"\b{re.escape(expected)}\b", first_line, re.IGNORECASE) is not None
    return correct, first_line


def score(template, examples, max_tokens=10, stop=None):
    """Fills {text} in the template with every example, sends it to the model, and counts the correct answers."""
    print("Prompt for the first example:\n" + template.replace("{text}", examples[0][0]) + "\n")
    options = {"num_predict": max_tokens, "temperature": 0}
    if stop:
        options["stop"] = stop
    correct, tokens = 0, 0
    for text, expected in examples:
        response = olm.generate(model=BASE_MODEL, prompt=template.replace("{text}", text), options=options)
        ok, answer = check(response.response, expected)
        correct += ok
        tokens += response.eval_count or 0  # an empty prompt generates nothing
        print(f"{'✓' if ok else '✗'} expected: {expected:<10} model: {answer[:70]}")
    print(f"\n{correct}/{len(examples)} correct, {tokens} tokens generated")

In [ ]:
# ten short movie reviews with their sentiment
REVIEWS = [
    ("A wonderful film with brilliant actors.", "positive"),
    ("The plot was boring and far too long.", "negative"),
    ("I laughed from start to finish.", "positive"),
    ("A complete waste of two hours.", "negative"),
    ("Not bad at all, I would watch it again.", "positive"),
    ("The ending made no sense, and the dialogue was awful.", "negative"),
    ("Beautiful music and a touching story.", "positive"),
    ("I fell asleep halfway through.", "negative"),
    ("It was not as good as everyone said.", "negative"),
    ("My kids loved it, and so did I.", "positive"),
]

In [ ]:
zero_shot = """Review: {text}
Sentiment:"""
score(zero_shot, REVIEWS)

In [ ]:
# TODO: write a one-shot and a few-shot template, and keep the placeholder {text}
one_shot = """"""
few_shot = """"""

score(one_shot, REVIEWS)
score(few_shot, REVIEWS)

## Part 3: From Few-Shot Prompts to Conversations

Writing a new text pattern for every task is tedious, and researchers found this really annoying. A more general pattern is to write *every* task as a **conversation between a user and an assistant**. A base model has seen many conversations during pretraining, such as forums, interviews, and chat logs, so we can show it this pattern with a few examples.

With the same trick, we can also show the model *how* to answer. [Wei et al. (2022)](https://proceedings.neurips.cc/paper_files/paper/2022/file/9d5609613524ecf4f15af0f7b31abca4-Paper-Conference.pdf) gave models few-shot examples in which the answers contain the reasoning steps before the result. They called this **chain-of-thought (CoT) prompting**:

![Chain-of-Thought Prompting](https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_04/imgs/cot.png)


As visible in this illustration from the paper of Wei et al. (2022), you can see that they apply few-shot prompting to incorporate the CoT reasoning into the model ([Wei et al., 2022](https://proceedings.neurips.cc/paper_files/paper/2022/file/9d5609613524ecf4f15af0f7b31abca4-Paper-Conference.pdf)).

Here is the example from the paper, written as a conversation. The example answer shows the reasoning steps:

In [ ]:
conversation = """The following is a conversation between a user and a helpful assistant. The assistant thinks step by step before it answers.

User: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
Assistant: Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11.

User: The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?
Assistant:"""

response = olm.generate(model=BASE_MODEL, prompt=conversation, options={"num_predict": 150, "temperature": 0, "seed": 42})
print(response.response)

Look at the end of the output. The model probably did not stop after its answer: it also wrote the next `User:` line, maybe even a whole new question. The most likely continuation of a conversation is more conversation, so the base model plays **both roles**.

### Your turn

Now you build the chain-of-thought prompt yourself, for the six word problems below.

1. Write a conversation template with at least one step-by-step example of your own. End the example answer with `The answer is ...`, because `score()` looks for the answer behind these words. End the template with `User: {text}` and an open `Assistant:`.
2. Copy your template and remove the reasoning from the example answer, so that only `The answer is ...` remains. Which version scores better?

Small models often copy the *form* of the steps without getting the arithmetic right, so don't be surprised.

In [ ]:
# six word problems with their answers
WORD_PROBLEMS = [
    ("A baker makes 4 trays with 6 cookies on each tray. He sells 10 cookies. How many cookies are left?", "14"),
    ("Lisa has 3 boxes with 5 pencils each. She gives 4 pencils to her brother. How many pencils does she have now?", "11"),
    ("A bus has 20 passengers. At the first stop, 7 get off and 3 get on. How many passengers are on the bus now?", "16"),
    ("Tom reads 8 pages every day. How many pages does he read in one week?", "56"),
    ("A shirt costs 15 euros. Anna buys 2 shirts and pays with a 50 euro note. How much change does she get?", "20"),
    ("There are 12 eggs in a carton. Mia uses 5 eggs for a cake and 2 for breakfast. How many eggs are left?", "5"),
]

In [ ]:
# TODO 1: your conversation template with a step-by-step example
cot_template = """"""
score(cot_template, WORD_PROBLEMS, max_tokens=150)

# TODO 2: the same template without the reasoning steps
no_cot_template = """"""
score(no_cot_template, WORD_PROBLEMS, max_tokens=150)

## Part 4: How Does the Model Know When to Stop?

A language model never "decides" that it is done. Generation stops in one of three cases:

1. The **maximum number of tokens** is reached (`max_tokens`, called `num_predict` in Ollama). `done_reason` is then `length`.
2. The model generates a **special end token**. A base model learned `<|end_of_text|>` for the end of a document, but it rarely predicts it in the middle of a conversation. `done_reason` is then `stop`.
3. The application finds a **stop sequence** in the generated text and cuts it there. This is also reported as `stop`.

With a stop sequence in the options, we tell Ollama to cut the text as soon as the model starts the next user turn:

In [ ]:
response = olm.generate(
    model=BASE_MODEL,
    prompt=conversation,
    options={"num_predict": 150, "temperature": 0, "stop": ["\nUser:"]},
)
print(response.response)
print("stopped because of:", response.done_reason)

The model still *wants* to write the user's line, and Ollama just cuts it. A stop sequence is a crude fix: what if a user writes "User:" in their own message?

The better fix needs training: teach the model to end every answer with a special **end-of-turn token**, and stop as soon as it generates this token. This is one of the things chat models learn.

### Your turn

1. Run the few-shot math prompt from Part 2 again, but with `stop=["\n"]`. **Predict** the output and the `done_reason`.
2. Score your best sentiment template from Part 2 again, now with `stop=["\n"]`. The accuracy should stay the same, but compare the model's answers and the number of generated tokens. Why is this useful in an application?
3. Most of you know ChatGPT. What is different between the answers of our base model and those of ChatGPT? Write your answer as a comment below and share it with the seminar.

In [ ]:
# TODO 1: the few-shot math prompt from Part 2 (few_shot_prompt) with a stop sequence


# TODO 2: your best sentiment template from Part 2, now with a stop sequence
score(few_shot, REVIEWS, stop=["\n"])

In [ ]:
# Write here your answer to the question: What is different in this model's answers compared to those of ChatGPT?
#

## Questions?

Now is the time for questions! Is anything about base models, few-shot patterns, or stopping unclear? Ask now, because the next notebook builds on it.

## What You Learned

* A language model sees **one sequence of tokens** and continues it. Everything else is formatting around this text.
* For a **base model**, what you send is what the model sees: Ollama's template is only `{{ .Prompt }}`.
* **Zero-, one-, and few-shot prompting** and **chain-of-thought** are patterns inside this text. The model continues the pattern, and it does not stop on its own.
* Generation stops at the **maximum number of tokens**, at a **stop sequence**, or at a **special end token**.
* A **prompt template** has a placeholder for the input. With a few examples with known answers, you can measure which template works best, and small changes can change the score.
* A stop sequence is a crude fix. Chat models learn to end their answer with an **end-of-turn token** instead.

In the next notebook, **4.02 - Prompt Template Engineering**, we look at exactly this: the chat templates that turn a base model into a chat model, the system prompt, and reasoning models.